#### Demo - Translator Agents as Tools

This example shows the agents-as-tools pattern. The frontline agent receives a user message and
then picks which agents to call, as tools. In this case, it picks from a set of translation
agents.


In [1]:
# Necessary imports
from dotenv import load_dotenv
# ItemHelpers, MessageOutputItem, Importing helper utilities to format structured agent outputs (like text messages or other content)
from agents import Agent, Runner, trace, ItemHelpers, MessageOutputItem


In [2]:

load_dotenv()

True

In [3]:
# Defining Spanish Agent
spanish_agent = Agent(
    name="spanish_agent",
    instructions="You translate the user's message to Spanish",
    model='gpt-4o-mini'
)


In [4]:
# Defining French Agent
french_agent = Agent(
    name="french_agent",
    instructions="You translate the user's message to French",
    model='gpt-4o-mini'
)

In [5]:
#  Defining Italian Agent
italian_agent = Agent(
    name="italian_agent",
    instructions="You translate the user's message to Italian",
    model='gpt-4o-mini'
)

In [6]:
# Orchester Agent directs to the respective Tools
orchestrator_agent = Agent(
    name="orchestrator_agent",
    instructions=(
        "You are a translation agent. You use the tools given to you to translate."
        "If asked for multiple translations, you call the relevant tools in order."
        "You never translate on your own, you always use the provided tools."
    ),
    model='gpt-4o-mini',
    tools=[
        spanish_agent.as_tool(
            tool_name="translate_to_spanish",
            tool_description="Translate the user's message to Spanish",
        ),
        french_agent.as_tool(
            tool_name="translate_to_french",
            tool_description="Translate the user's message to French",
        ),
        italian_agent.as_tool(
            tool_name="translate_to_italian",
            tool_description="Translate the user's message to Italian",
        ),
    ],
)

**instructions:** 
* Tells the agent what to do (internal behavior). 
* The agent itself to know its behavior/policy.

**tool_description:** 
* Describes the agent when registered as a tool.
* The orchestrator agent (or any agent using tools) to choose the right tool.

In [ ]:
# Calling the orchestrator_agent
result = await Runner.run(orchestrator_agent, input="Say 'Hello, how are you?' in Spanish.")
print(result.final_output)

In Spanish, it's: "Hola, ¿cómo estás?"


In [ ]:
# Calling the orchestrator_agent for multiple translation
result = await Runner.run(orchestrator_agent, input="Translate Welcome to AI Agents Learning into Spanish and French")
print(result.final_output)

Here are the translations:

- **Spanish:** Bienvenido a AI Agents Learning.
- **French:** Bienvenue chez AI Agents Learning.


In [ ]:
# Evaluator Agent, use other models
synthesizer_agent = Agent(
    name="synthesizer_agent",
    instructions="You inspect translations, correct them if needed, and produce a final concatenated response.",
    model='gpt-5-nano'
)

In [16]:
msg = "Translate Welcome to AI Agents Learning into Spanish and French"

In [17]:
# Run the entire orchestration in a single trace
with trace("Orchestrator evaluator"):
    orchestrator_result = await Runner.run(orchestrator_agent, msg)

# Loop through all newly generated items (outputs) from the orchestrator agent
    for item in orchestrator_result.new_items:
        # Check if the item is a message-type output 
        if isinstance(item, MessageOutputItem):
             # Use the helper method to safely extract plain text from the message output
            text = ItemHelpers.text_message_output(item)
             # If text content was successfully extracted, print it to the console
            if text:
                print(f"  - Translation step: {text}")

    # Passing the orchestrator_result as input to evaluate through synthesizer agent
    synthesizer_result = await Runner.run(
        synthesizer_agent, orchestrator_result.to_input_list()
        )

    print(f"\n\nFinal response:\n{synthesizer_result.final_output}")


  - Translation step: Here are the translations:

- Spanish: **Bienvenido a AI Agents Learning.**
- French: **Bienvenue dans l'apprentissage des agents IA.**


Final response:
Here are the translations:

- Spanish: Bienvenido a AI Agents Learning.
- French: Bienvenue dans l'apprentissage des agents d'IA.
